# AxiomRunner T4 acceptance run

This notebook provisions a fresh Google Colab GPU runtime, verifies the existing Docker sandbox, runs the unchanged ten-file acceptance corpus with local `qwen3:8b`, and downloads a redacted evidence bundle. Run every cell in order. Do not upload credentials or generated solutions.

## 1. Confirm the GPU and configure the run

Before running this cell, choose **Runtime > Change runtime type > T4 GPU**. The cell stops before installing anything if no suitable GPU is attached.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

REPOSITORY = "https://github.com/MutugiD/AxiomRunner.git"
REF = "main"
MODEL = "qwen3:8b"
REQUIRE_T4 = True
WORKSPACE = Path("/content/AxiomRunner")
CORPUS = Path("/content/challengebox-samples")
RESULTS = Path("/content/axiomrunner-t4-results")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"GPU: {gpu}")
if REQUIRE_T4 and "T4" not in gpu:
    raise RuntimeError("A T4 was requested but Colab assigned a different GPU")

## 2. Install and start the isolated runtime dependencies

Docker is used only for candidate verification. Ollama remains bound to loopback and cloud features are disabled. All installations live in this disposable Colab VM.

In [ ]:
install_env = os.environ.copy()
install_env["DEBIAN_FRONTEND"] = "noninteractive"
subprocess.run(["apt-get", "update", "-qq"], check=True, env=install_env)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "docker.io", "curl"],
    check=True,
    env=install_env,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv==0.10.9"],
    check=True,
)
subprocess.run(
    ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
    check=True,
)


def command_ok(command: list[str]) -> bool:
    return subprocess.run(command, capture_output=True, text=True).returncode == 0


if not command_ok(["docker", "info"]):
    with Path("/content/dockerd.log").open("ab", buffering=0) as docker_log:
        docker_process = subprocess.Popen(
            [
                "dockerd",
                "--host=unix:///var/run/docker.sock",
                "--storage-driver=vfs",
                "--iptables=false",
                "--bridge=none",
            ],
            stdout=docker_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    for _ in range(45):
        if command_ok(["docker", "info"]):
            break
        time.sleep(1)
    else:
        raise RuntimeError("Docker did not start; inspect /content/dockerd.log")
print(
    subprocess.run(
        ["docker", "version", "--format", "{{.Server.Version}}"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout
)

## 3. Clone the exact application and start the local model

The resolved Git commit is recorded later in `environment.json`. The model is warmed before GPU residency is checked.

In [ ]:
if WORKSPACE.exists():
    raise RuntimeError(
        "/content/AxiomRunner already exists; use a fresh runtime for an official run"
    )
subprocess.run(
    ["git", "clone", "--branch", REF, "--single-branch", REPOSITORY, str(WORKSPACE)], check=True
)
subprocess.run(["uv", "python", "install", "3.12"], check=True, cwd=WORKSPACE)
subprocess.run(
    ["uv", "sync", "--locked", "--all-groups", "--python", "3.12"],
    check=True,
    cwd=WORKSPACE,
)
resolved_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    check=True,
    cwd=WORKSPACE,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Resolved commit: {resolved_commit}")

ollama_env = os.environ.copy()
ollama_env.update(
    {
        "OLLAMA_HOST": "127.0.0.1:11434",
        "OLLAMA_NO_CLOUD": "1",
        "OLLAMA_FLASH_ATTENTION": "1",
        "OLLAMA_CONTEXT_LENGTH": "8192",
    }
)
ollama_log_path = Path("/content/ollama.log")
ollama_log = ollama_log_path.open("ab", buffering=0)
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    env=ollama_env,
    start_new_session=True,
)
for _ in range(45):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2):
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; inspect /content/ollama.log")
subprocess.run(["ollama", "pull", MODEL], check=True, env=ollama_env)
warmup = json.dumps(
    {
        "model": MODEL,
        "stream": False,
        "think": False,
        "keep_alive": -1,
        "messages": [{"role": "user", "content": "Reply with OK."}],
        "options": {"num_predict": 8, "temperature": 0, "seed": 7},
    }
).encode()
request = urllib.request.Request(
    "http://127.0.0.1:11434/api/chat",
    data=warmup,
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(request, timeout=180) as response:
    json.load(response)
ollama_ps = subprocess.run(
    ["ollama", "ps"],
    check=True,
    env=ollama_env,
    capture_output=True,
    text=True,
).stdout.strip()
print(ollama_ps)
if "GPU" not in ollama_ps:
    raise RuntimeError("Ollama did not report GPU residency; stop before benchmarking")

## 4. Verify AxiomRunner and its Docker trust boundary

This preflight is mandatory. A run without the sandbox controls is not acceptance evidence.

In [ ]:
run_env = os.environ.copy()
run_env.update(
    {
        "AXIOMRUNNER_MODEL": MODEL,
        "AXIOMRUNNER_OLLAMA_HOST": "http://127.0.0.1:11434",
        "AXIOMRUNNER_SEED": "7",
        "AXIOMRUNNER_CANDIDATES": "1",
        "AXIOMRUNNER_REPAIRS": "1",
    }
)
subprocess.run(["uv", "run", "axiomrunner", "doctor"], check=True, cwd=WORKSPACE, env=run_env)
sandbox_image = (
    "python:3.12-slim@sha256:78387bc3881b8273120a12ebe6c1ab22b018ccc2c9adf565ae1ac9b536e184ea"
)
subprocess.run(["docker", "pull", sandbox_image], check=True)
test_env = run_env.copy()
test_env["AXIOMRUNNER_DOCKER_TEST"] = "1"
subprocess.run(
    ["uv", "run", "pytest", "-m", "docker", "--no-cov", "-q"],
    check=True,
    cwd=WORKSPACE,
    env=test_env,
)

## 5. Upload and validate the private corpus

Select the ten unchanged JSON files from `ChallengeBox/samples`. They remain in the temporary VM and are excluded from the downloaded evidence bundle.

In [ ]:
from google.colab import files

uploaded = files.upload()
if CORPUS.exists():
    shutil.rmtree(CORPUS)
CORPUS.mkdir(parents=True)
for raw_name, content in uploaded.items():
    name = Path(raw_name).name
    if Path(name).suffix.lower() != ".json":
        raise ValueError(f"Only JSON files are allowed: {name}")
    (CORPUS / name).write_bytes(content)

documents = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(CORPUS.glob("*.json"))]
languages = Counter(document.get("language") for document in documents)
problem_ids = [document.get("problem_id") for document in documents]
python_entrypoints = {
    document.get("entrypoint") for document in documents if document.get("language") == "python"
}
expected_entrypoints = {
    "simulate_writes",
    "track_indicator",
    "refresh_references",
    "validate_build",
    "normalize_protection",
    "capture_binders",
}
if len(documents) != 10 or languages != Counter({"python": 6, "rust": 4}):
    raise ValueError(
        f"Expected 10 files (6 Python, 4 Rust), received {len(documents)}: {languages}"
    )
if len(set(problem_ids)) != 10 or None in problem_ids:
    raise ValueError("Problem IDs must be present and unique")
if python_entrypoints != expected_entrypoints:
    raise ValueError(f"Unexpected Python entrypoints: {sorted(python_entrypoints)}")
if any(document.get("deadline_s") != 300.0 for document in documents):
    raise ValueError("Every corpus item must retain its 300-second deadline")
print(f"Corpus accepted: {languages['python']} Python and {languages['rust']} Rust files")

## 6. Run the complete benchmark

Keep this tab connected. The cell streams progress and records it. Exit code `4` means the release gate failed, but evidence collection continues.

In [ ]:
if RESULTS.exists():
    shutil.rmtree(RESULTS)
RESULTS.mkdir(parents=True)
report_path = RESULTS / "benchmark.json"
log_path = RESULTS / "benchmark.log"
command = [
    "uv",
    "run",
    "axiomrunner",
    "benchmark",
    str(CORPUS),
    "--report",
    str(report_path),
]
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command,
        cwd=WORKSPACE,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    benchmark_exit_code = process.wait()
if benchmark_exit_code not in {0, 4}:
    raise RuntimeError(f"Benchmark infrastructure failed with exit code {benchmark_exit_code}")
if not report_path.is_file():
    raise RuntimeError("Benchmark did not produce its redacted report")
print(f"Benchmark exit code: {benchmark_exit_code}")

## 7. Summarize and download the evidence

Send the downloaded ZIP for review. Do not commit it to the repository.

In [ ]:
report = json.loads(report_path.read_text(encoding="utf-8"))


def assert_redacted(value: object) -> None:
    if isinstance(value, dict):
        forbidden = {"source", "solution_source", "candidate_source", "prompt", "response"}
        present = forbidden.intersection(str(key).lower() for key in value)
        if present:
            raise ValueError(f"Source-bearing fields found in report: {sorted(present)}")
        for nested in value.values():
            assert_redacted(nested)
    elif isinstance(value, list):
        for nested in value:
            assert_redacted(nested)


assert_redacted(report)
records = report.get("records", [])
gate_passed = (
    report.get("successful") is True
    and report.get("solved") == 6
    and report.get("rejected_unsupported") == 4
    and len(records) == 10
    and all(record.get("accepted") is True for record in records)
)


def capture(
    command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None
) -> str:
    completed = subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)
    return (completed.stdout or completed.stderr).strip()


environment = {
    "captured_at_utc": datetime.now(UTC).isoformat(),
    "repository": REPOSITORY,
    "ref": REF,
    "commit": resolved_commit,
    "gpu": gpu,
    "python": capture(["uv", "run", "python", "--version"], cwd=WORKSPACE),
    "uv": capture(["uv", "--version"]),
    "ollama": capture(["ollama", "--version"], env=ollama_env),
    "ollama_process": ollama_ps,
    "docker": capture(["docker", "version", "--format", "{{.Server.Version}}"]),
    "model": MODEL,
    "seed": 7,
    "candidate_limit": 1,
    "repair_limit": 1,
    "benchmark_exit_code": benchmark_exit_code,
}
(RESULTS / "environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

lines = [
    "# AxiomRunner T4 Acceptance Summary",
    "",
    f"**Release gate: {'PASS' if gate_passed else 'FAIL'}**",
    "",
    f"- Commit: `{resolved_commit}`",
    f"- GPU: {gpu}",
    f"- Model: `{MODEL}`",
    f"- Aggregate elapsed time: {float(report.get('elapsed_s', 0)):.2f} seconds",
    f"- Python challenges solved: {report.get('solved', 0)}/6",
    f"- Rust challenges rejected before inference: {report.get('rejected_unsupported', 0)}/4",
    "",
    "| File | Status | Accepted | Seconds | Candidates | Repairs | Checks |",
    "| --- | --- | --- | ---: | ---: | ---: | ---: |",
]
for record in records:
    lines.append(
        f"| `{record.get('file', '')}` | {record.get('status', '')} | "
        f"{record.get('accepted', False)} | {float(record.get('elapsed_s', 0)):.2f} | "
        f"{record.get('candidates', 0)} | {record.get('repairs', 0)} | {record.get('checks', 0)} |"
    )
lines.extend(
    [
        "",
        "The v0.1.0 tag is eligible for a separate release change only "
        "when this gate reports PASS.",
    ]
)
summary_path = RESULTS / "summary.md"
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

ollama_tail = "\n".join(
    ollama_log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-200:]
)
(RESULTS / "ollama.log.tail.txt").write_text(ollama_tail + "\n", encoding="utf-8")
bundle = Path("/content/axiomrunner-t4-evidence.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.iterdir()):
        archive.write(path, arcname=path.name)
print(summary_path.read_text(encoding="utf-8"))
files.download(str(bundle))